DECISION TREE ASSIGNMENT

Question 1: What is a Decision Tree, and how does it work in the context of classification?

Ans -
A Decision Tree is a supervised machine-learning algorithm that represents decisions as a tree-like structure. It contains a root node, internal decision nodes, branches, and leaf nodes. In classification, each leaf predicts a class label, such as “disease” or “no disease.”
How it works:
1. The algorithm begins with the training data at the root node.
2. It considers candidate features and split points, then selects a split that best separates the classes according to a criterion such as Gini impurity or entropy/information gain.
3. The data is divided into child nodes according to the selected rule.
4. The process is repeated recursively until a stopping condition is reached, such as a maximum depth, too few samples to split, or sufficiently pure leaves.
5. To classify a new observation, the model follows the rules from the root to a leaf. The leaf’s class (or class probabilities) becomes the prediction.
For example, a tree could first ask whether a flower’s petal length is below a threshold, then use petal width to distinguish between flower species. Decision Trees can model nonlinear relationships and are relatively easy to interpret, but an unconstrained tree can overfit the training data.


Question 2: Explain the concepts of Gini Impurity and Entropy as impurity measures. How do they impact the splits in a Decision Tree?

Ans -
Gini impurity and entropy measure how mixed the class labels are in a node. A node containing only one class has impurity 0; a node with a mixture of classes has higher impurity.
Gini impurity:
Gini = 1 − Σ(pᵢ²), where pᵢ is the proportion of observations belonging to class i in the node. It is 0 for a pure node. For a fixed number of classes, it is largest when the classes are evenly distributed. The Gini criterion is commonly used by CART Decision Trees.
Entropy:
Entropy = −Σ(pᵢ log₂ pᵢ), where pᵢ is the proportion of observations in class i. A zero-probability term is treated as 0. Entropy is 0 for a pure node and increases as the class distribution becomes more mixed.
Impact on splitting:
For each candidate split, the algorithm calculates the weighted impurity of the resulting child nodes. It prefers the split that produces the greatest reduction in impurity (for entropy, this reduction is commonly called information gain). Gini and entropy often select similar splits, although their rankings can differ slightly. Gini is generally computationally simpler, while entropy has an information-theoretic interpretation. Neither measure guarantees the best generalization on unseen data, so performance should be validated on held-out data.


Question 3: What is the difference between Pre-Pruning and Post-Pruning in Decision Trees? Give one practical advantage of using each.

Ans -
Pre-pruning (early stopping) limits tree growth while the tree is being built. Examples include setting max_depth, min_samples_split, min_samples_leaf, or a minimum impurity decrease. Practical advantage: it can reduce training time and prevent a tree from becoming unnecessarily complex.
Post-pruning allows a larger tree to grow first and then removes branches that contribute little to predictive performance. Cost-complexity pruning, controlled by ccp_alpha in scikit-learn, is one example. Practical advantage: the model can first discover detailed patterns and then remove branches that are not worth their added complexity.
In short, pre-pruning stops growth early; post-pruning simplifies an already-grown tree. Both can reduce overfitting. Their settings should be selected using validation data or cross-validation rather than by looking only at training accuracy.


Question 4: What is Information Gain in Decision Trees, and why is it important for choosing the best split?

Ans -
Information Gain measures how much a split reduces uncertainty about the class labels. For a parent node S and child subsets Sᵥ, it is:
Information Gain = H(S) − Σᵥ (|Sᵥ| / |S|) H(Sᵥ)
Here, H(S) is the entropy of the parent node, |Sᵥ| is the number of samples in child v, and |S| is the number of samples in the parent. The second term is the weighted average entropy after splitting.
A split with higher information gain creates child nodes that are, overall, more class-pure. This helps the tree choose a feature and threshold that separate the training classes effectively. The process is applied recursively at later nodes. Information Gain is not the only possible split criterion: CART can instead minimize weighted Gini impurity. A high training information gain does not by itself guarantee strong test performance, and some criteria can favor features with many possible values, so validation remains important.


Question 5: What are some common real-world applications of Decision Trees, and what are their main advantages and limitations?

Ans -
Common applications:
• Healthcare: screening or classifying disease risk using patient measurements and records.
• Banking and finance: credit-risk classification, loan default prediction, and fraud detection.
• Retail and marketing: customer segmentation, churn prediction, and purchase-response modelling.
• Manufacturing: identifying likely equipment failures from sensor and maintenance data.
• Telecommunications: predicting customer churn.
• Environmental and scientific work: classifying observations using measured attributes.
Advantages:
1. The rules can be visualized and explained to non-technical users.
2. They can model nonlinear relationships and feature interactions.
3. They generally need less feature scaling than distance-based models.
4. They can be used for classification and regression.
5. Feature importance estimates can provide a rough indication of which features the fitted tree used.
Limitations:
1. Deep trees can overfit, especially with small or noisy datasets.
2. Small changes in training data can produce a different tree.
3. A single tree may be less accurate than ensemble methods such as Random Forests or Gradient Boosting.
4. Greedy split selection does not necessarily find a globally optimal tree.
5. Impurity-based feature importance can be biased and should not be interpreted as causal evidence.
6. Missing values, categorical variables, class imbalance, and unequal error costs need deliberate handling; scikit-learn’s standard DecisionTree models require preprocessing for many such cases.
7. In high-stakes settings, predictions require validation, fairness checks, privacy safeguards, and human oversight.


In [1]:
#Question 6: Write a Python program to load the Iris Dataset, train a Decision Tree Classifier using the Gini criterion, and print the model’s accuracy and feature importances.

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the Iris classification dataset
iris = load_iris()
X, y = iris.data, iris.target

# Keep class proportions similar in training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train a Decision Tree using Gini impurity
model = DecisionTreeClassifier(criterion="gini", random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Feature importances:")
for name, importance in zip(iris.feature_names, model.feature_importances_):
    print(f"{name}: {importance:.4f}")



Accuracy: 0.9333
Feature importances:
sepal length (cm): 0.0062
sepal width (cm): 0.0292
petal length (cm): 0.5586
petal width (cm): 0.4060


In [2]:
#Question 7: Write a Python program to load the Iris Dataset, train a Decision Tree Classifier with max_depth=3 and compare its accuracy to a fully-grown tree.


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# A depth-limited tree
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_tree.fit(X_train, y_train)

# A fully-grown tree: max_depth=None allows growth until other stopping rules apply
full_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
full_tree.fit(X_train, y_train)

shallow_accuracy = accuracy_score(y_test, shallow_tree.predict(X_test))
full_accuracy = accuracy_score(y_test, full_tree.predict(X_test))

print(f"max_depth=3 accuracy: {shallow_accuracy:.4f}")
print(f"Fully-grown tree accuracy: {full_accuracy:.4f}")



max_depth=3 accuracy: 0.9667
Fully-grown tree accuracy: 0.9333


In [3]:
#Question 9: Write a Python program to load the Iris Dataset, tune the Decision Tree’s max_depth and min_samples_split using GridSearchCV, and print the best parameters and resulting model accuracy.


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

base_model = DecisionTreeClassifier(random_state=42)
param_grid = {
    "max_depth": [None, 2, 3, 4, 5, 6],
    "min_samples_split": [2, 5, 10]
}

search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
search.fit(X_train, y_train)

best_model = search.best_estimator_
test_accuracy = accuracy_score(y_test, best_model.predict(X_test))

print("Best parameters:", search.best_params_)
print(f"Best cross-validation accuracy: {search.best_score_:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")



Best parameters: {'max_depth': None, 'min_samples_split': 2}
Best cross-validation accuracy: 0.9417
Test accuracy: 0.9333


Question 10: Imagine you’re working as a data scientist for a healthcare company that wants to predict whether a patient has a certain disease. You have a large dataset with mixed data types and some missing values. Explain the step-by-step process you would follow to handle missing values, encode categorical features, train a Decision Tree model, tune its hyperparameters, evaluate performance, and describe its business value.

Ans -
1. Define the prediction task and target.
Work with clinical and domain experts to define the disease outcome, prediction time, intended users, and action that follows a prediction. Specify the positive class and avoid using information that would not be available at prediction time (data leakage). Establish privacy, consent, security, and governance requirements.
2. Inspect and split the data.
Check data types, ranges, duplicates, label quality, class balance, missingness, and possible outliers. Split data into training, validation (or cross-validation), and final test sets before fitting preprocessing steps. If records from the same patient repeat, keep that patient’s records in one split. If the model predicts future cases, consider a time-based split. This reduces leakage and produces a more realistic evaluation.
3. Handle missing values.
Investigate why values are missing and whether missingness itself carries meaningful information. Fit imputation only on the training data, then apply the fitted transformer to validation and test data. For numeric features, use a median imputer as a robust baseline; for categorical features, use the most-frequent category or an explicit “Missing” category. Consider a missing-indicator feature when clinically justified. Do not automatically drop every row with missing data, since that can waste information or introduce bias. Check whether imputation distorts important clinical patterns.
4. Encode categorical features.
For nominal categories with no natural order, use one-hot encoding. For genuinely ordered categories, use a carefully defined ordinal encoding. Group rare categories only when justified and handle previously unseen categories safely. Standard scikit-learn DecisionTreeClassifier workflows generally require categorical variables to be converted to numeric representations first. Keep all learned preprocessing inside a pipeline so cross-validation fits it separately within each training fold.
5. Train a baseline Decision Tree.
Use a classification tree, for example DecisionTreeClassifier(class_weight="balanced", random_state=42) if class imbalance makes that appropriate. Compare against a simple baseline and inspect class proportions. A tree is interpretable, but it can overfit, so do not assume the most complex tree is best.
6. Build a leakage-safe preprocessing and modelling pipeline.
Use a ColumnTransformer to impute numeric columns and impute plus one-hot encode categorical columns. Combine it with the classifier in a scikit-learn Pipeline. This ensures preprocessing is learned from the training folds only and that the same transformations are used at prediction time.
7. Tune hyperparameters.
Use GridSearchCV or RandomizedSearchCV with stratified cross-validation for classification, where appropriate. Tune max_depth, min_samples_split, min_samples_leaf, criterion, and ccp_alpha. Select a scoring metric that reflects the healthcare objective, such as recall/sensitivity when missing a true case is especially costly, or a precision-recall metric when false alarms are also a major concern. Do not tune against the final test set.
8. Evaluate on held-out data.
Report a confusion matrix, sensitivity/recall, specificity, precision, F1 score, ROC-AUC and/or precision-recall AUC, and calibration where relevant. Accuracy alone can be misleading when disease cases are rare. Choose a decision threshold with clinicians based on the relative harms of false negatives and false positives. Check confidence intervals, performance across clinically relevant demographic and patient groups, and robustness to changes in hospitals or time periods. Evaluate the final selected pipeline once on the untouched test set.
9. Interpret, validate, and monitor.
Review decision rules and feature importance with clinicians; use additional explanation methods cautiously. Feature importance is not causality. Before deployment, conduct clinical, safety, fairness, privacy, and workflow validation, and retain human review. Monitor data drift, missingness changes, false-negative rates, and model performance after deployment, and retrain only under controlled governance.
Business value:
A validated model could help prioritize patients for further assessment, support earlier detection, allocate clinical resources, and make screening more consistent. It may reduce avoidable delays and support operational planning. These benefits are potential, not guaranteed: they depend on external validation, appropriate clinical workflows, fairness, patient safety, and evidence that using the model improves outcomes. The model should support qualified healthcare professionals rather than independently diagnose patients.
